# Phase 5d - Post-hoc calibration
Validation showed small, **stable** biases in the final pipeline:
- service time ~0.6 min too low on average - the log-target half of the blend predicts the conditional *median*, which sits below the mean for this right-skewed target;
- the late probability ~0.4 points too low.

Three corrections, all fitted on validation-fold predictions (never on the rows they are scored on):

| Correction | Parameter |
|---|---|
| Rescale the log-target bag so its mean matches the actual mean | `log_scale = sum(y) / sum(log-bag prediction)` |
| Rescale Style and Tech service per brand | `brand_scale[b] = sum(y_b) / sum(pred_b)`; Fresh is left alone (scaling it hurt MAE) |
| Shift the lateness log-odds | `late_logit_shift` minimising log-loss |

**Honest test:** leave-one-fold-out - fit on two folds, score on the third, for every fold.

In [ ]:
import sys, json, time; sys.path.append("../../tools/task1")
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from task1_common import *
from task1_validation import FOLDS, service_metrics, late_metrics
import task1_models as tm
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
t = load_task1_tables()

## 1. Uncalibrated validation predictions, with model parts
Retrains the pipeline on each fold (~5 min) the first time, then caches to `reports/metrics/task1/task1_calibration_fold_parts.pkl`.

In [ ]:
CACHE = METRICS / 'task1_calibration_fold_parts.pkl'
FD = PROCESSED / 'folds'
FEATURES = json.load(open(FD / 'folds_meta.json'))['features']
arrivals = pd.read_pickle(INTERIM / 'train_actuals_diagnostic.pkl')[['delivery_id', 'arrival_time_m']]
arrival_for = lambda df: df[['delivery_id']].merge(arrivals, on='delivery_id', how='left', validate='one_to_one').arrival_time_m

if CACHE.exists():
    parts = pd.read_pickle(CACHE); print('loaded cache', parts.shape)
else:
    out = []
    for name, _, _ in FOLDS:
        t0 = time.time()
        X_tr, X_va = pd.read_pickle(FD / f'{name}_train.pkl'), pd.read_pickle(FD / f'{name}_valid.pkl')
        m = tm.fit(X_tr, arrival_for(X_tr), FEATURES, t['traffic'], verbose=False)
        svc, late, p = tm.predict(m, X_va, t['traffic'], return_parts=True, calibrate=False)
        out.append(pd.DataFrame({'fold': name, 'brand': X_va.brand.astype(str).values, 'monsoon': X_va.monsoon.values,
                                 'y_svc': X_va.service_min.values, 'y_late': X_va.late.values, 'svc': svc, 'late': late,
                                 'svc_raw': p['svc_raw'], 'svc_log': p['svc_log'], 'svc_segment': p['svc_segment']}))
        print(f'{name}: {time.time() - t0:.0f}s')
    parts = pd.concat(out, ignore_index=True); parts.to_pickle(CACHE)
print(parts.groupby('fold').agg(service_bias=('svc', 'mean'), actual=('y_svc', 'mean'), pred_late=('late', 'mean'), late_rate=('y_late', 'mean')).round(4))

## 2. Leave-one-fold-out test

In [ ]:
def apply_cal(df, cal):
    seg = df.brand.isin(tm.SEGMENT_BRANDS).values
    svc = tm._combine_service(df.svc_raw.values.copy(), df.svc_log.values * cal['log_scale'], df.svc_segment.values, seg)
    for b, k in cal['brand_scale'].items():
        svc[df.brand.eq(b).values] *= k
    late = 1 / (1 + np.exp(-(tm._logit(df.late.values) + cal['late_logit_shift'])))
    return np.clip(svc, 1, None), np.clip(late, 1e-4, 1 - 1e-4)

rows, params = [], []
for f in parts.fold.unique():
    fit_on, score_on = parts[parts.fold != f], parts[parts.fold == f]
    cal = tm.fit_calibration(fit_on)
    params.append({'held_out': f, 'log_scale': cal['log_scale'], **{f'scale_{b}': k for b, k in cal['brand_scale'].items()}, 'late_logit_shift': cal['late_logit_shift']})
    s_cal, l_cal = apply_cal(score_on, cal)
    for label, s, l in [('before', score_on.svc.values, score_on.late.values), ('after', s_cal, l_cal)]:
        rows.append({'fold': f, 'version': label, **service_metrics(score_on.y_svc.values, s), 'bias': np.mean(s - score_on.y_svc.values),
                     **late_metrics(score_on.y_late.values, l)})
res = pd.DataFrame(rows)
print('parameters fitted on the other two folds (stable = trustworthy):'); print(pd.DataFrame(params).round(4).to_string(index=False))
cols = ['MAE', 'RMSE', 'bias', 'logloss', 'brier', 'AUC', 'ECE', 'mean_pred', 'rate']
print('\nmean over held-out folds:'); print(res.groupby('version')[cols].mean().loc[['before', 'after']].round(4))
print('\nper fold:'); print(res.pivot_table(index='fold', columns='version', values=['MAE', 'RMSE', 'logloss', 'brier'])[[(m, v) for m in ['MAE', 'RMSE', 'logloss', 'brier'] for v in ['before', 'after']]].round(4))

In [ ]:
b, a = res[res.version == 'before'].set_index('fold'), res[res.version == 'after'].set_index('fold')
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for i, m in enumerate(['MAE', 'RMSE', 'logloss', 'brier']):
    pd.DataFrame({'before': b[m], 'after': a[m]}).plot.bar(ax=ax[i], rot=0); ax[i].set_title(f'{m} (lower is better)')
    lo = min(b[m].min(), a[m].min()); ax[i].set_ylim(lo * 0.97, max(b[m].max(), a[m].max()) * 1.01)
plt.tight_layout(); plt.show()
print('improved in every fold:', {m: bool((a[m] <= b[m]).all()) for m in ['MAE', 'RMSE', 'logloss', 'brier']})

## 3. Fit on all three folds and attach to the final model
The final calibration uses all validation predictions. It is stored inside the saved model, so `tm.predict` applies it automatically. Re-running this cell is safe (it overwrites the calibration, never stacks it).

In [ ]:
cal = tm.fit_calibration(parts)
print('final calibration:', json.dumps(cal, indent=1))
path = MODELS / 'task1_models.joblib'
models = joblib.load(path)
models['calibration'] = cal
joblib.dump(models, path, compress=3)
with open(METRICS / 'task1_calibration.json', 'w') as f: json.dump(cal, f, indent=1)
res.to_csv(METRICS / 'task1_calibration_lofo.csv', index=False)
print('attached to', path, '| re-run 06_predict_submit.ipynb to regenerate the submission')

### Decision
Keep the calibration if the leave-one-fold-out table shows MAE not worse, RMSE / log-loss / Brier better, AUC unchanged, and parameters stable across folds. AUC is unaffected by design (all corrections are monotonic per brand), so the ranking quality of the model is preserved.